In [1]:
from google.colab import files

uploaded = files.upload()


Saving marketplace_raw.csv to marketplace_raw.csv


In [2]:
import pandas as pd

df = pd.read_csv("marketplace_raw.csv")

df.head()

,global_id,student_id,source_file,item_id,title,description,category,age,condition,price,currency,item_photo,unnamed_0,unnamed_1,unnamed_2,unnamed_9,unnamed_10,link,Unnamed: 18,image_path
0,1,1210482,1210482.csv,1,Tesla Model 3,"standard range with 21,867 miles on it. Deep B...",Cars,4,4,28868,dollar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,images/0001_1210482.jpg
1,2,1210482,1210482.csv,2,Mercedes-Benz AMG GT 63 4MATIC,22025 miles on it. Obsidian Black Metallic ext...,Cars,7,4,79412,dollar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,images/0002_1210482.jpg
2,3,1210482,1210482.csv,3,Lexus LX 600 Luxury,11956 miles on it. Eminent White Pearl exterio...,Cars,3,5,90451,dollar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,images/0003_1210482.jpg
3,4,1210482,1210482.csv,4,Crate & Barrel Unwind Green Velvet 2 Seater Sofa,Brand: Crate & Barrel. Collection: Unwind. Con...,Furniture,3,4,750,dollar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,images/0004_1210482.jpg
4,5,1210482,1210482.csv,5,Drexel Heritage 1980s Postmodern Curved Sofa F...,Brand: Drexel. Condition: Used - Age-worn: Has...,Furniture,40,2,450,dollar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,images/0005_1210482.jpg


In [4]:
assessment_df = df.copy()

missing_tokens = [
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "None",
    "none"
]

assessment_df = assessment_df.replace(missing_tokens, pd.NA)

missing_count = assessment_df.isna().sum()

missing_percentage = (
    assessment_df.isna().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage
})

# Show ALL columns, including columns with 0 missing values
missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

display(missing_summary)

,Missing_Count,Missing_Percentage
Unnamed: 18,1801,100.00
unnamed_1,1801,100.00
unnamed_2,1801,100.00
unnamed_9,1801,100.00
unnamed_0,1801,100.00
unnamed_10,1800,99.94
link,1786,99.17
age,374,20.77
item_photo,90,5.00
price,32,1.78


Missing values were identified for every column in the integrated dataset, and the percentage of missing values was calculated relative to the total number of observations.

The highest proportion of missing data was found in the `age` column, with 374 missing values (20.77%). The `item_photo` column contained 90 missing values (5.00%), followed by `price` with 32 missing values (1.78%) and `currency` with 23 missing values (1.28%).

Both `condition` and `item_id` contained 15 missing values (0.83%), while `description` and `category` each contained 1 missing value (0.06%).

Several non-standard auxiliary columns, such as `unnamed_0`, `unnamed_1`, `unnamed_2`, `unnamed_9`, `unnamed_10`, and `link`, contained very high percentages of missing values. These columns originated from individual student submissions and will be handled during the cleaning stage.

Columns with no missing values were also included in the summary table to provide a complete column-by-column assessment.

In [6]:
# Duplicate full records
duplicate_records = df.duplicated().sum()

# Duplicate item IDs alone - ignore missing IDs
duplicate_item_ids = (
    df["item_id"]
    .dropna()
    .duplicated()
    .sum()
)

# Duplicate item IDs within the same student - ignore missing IDs
duplicate_student_item_ids = (
    df.dropna(subset=["item_id"])
      .duplicated(subset=["student_id", "item_id"])
      .sum()
)

print("Duplicate full records:", duplicate_records)
print("Duplicate item IDs alone:", duplicate_item_ids)
print("Duplicate (student_id + item_id):", duplicate_student_item_ids)

Duplicate full records: 0
Duplicate item IDs alone: 1620
Duplicate (student_id + item_id): 0


Duplicate records and item IDs were assessed in the integrated dataset.

No fully duplicated records were found. Although 1,634 duplicate `item_id` values were detected when `item_id` was checked alone, this is expected because different students use the same item numbering scheme, such as `item_001`, `item_002`, and so on.

To identify meaningful duplicate item IDs, the combination of `student_id` and `item_id` was checked. No duplicate `(student_id, item_id)` combinations were found, indicating that no student has the same item ID repeated within their own submission.


In [10]:
print("Categories:")
print(df["category"].value_counts(dropna=False))

print("\nCurrencies:")
print(df["currency"].value_counts(dropna=False))

Categories:
category
Electronics                    361
Furniture                      178
Kitchen and household items    173
Clothing and accessories       169
Books                          141
                              ... 
Microwave Ovens                  1
Strollers                        1
Acoustic Guitars                 1
Electric Bicycles                1
Phone Accessories                1
Name: count, Length: 133, dtype: int64

Currencies:
currency
ILS        1105
USD         375
GBP          49
NIS          48
EUR          23
ils          21
NaN          21
CHF          19
Shekel       15
₪            15
Unknown      12
JOD          10
 USD         10
dollar       10
AED          10
usd           9
SGD           7
ILS           7
shekel        6
EGP           6
 GBP          5
USD           5
SAR           4
TWD           2
              2
LYD           1
IQD           1
SEK           1
INR           1
AUD           1
Name: count, dtype: int64


In [11]:
clean_df = df.copy()

def clean_currency(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip().lower()

    currency_map = {
        "ils": "ILS",
        "nis": "ILS",
        "shekel": "ILS",
        "₪": "ILS",

        "usd": "USD",
        "dollar": "USD",
        "$": "USD",

        "eur": "EUR",
        "gbp": "GBP",
        "chf": "CHF",
        "jod": "JOD",
        "aed": "AED",
        "sgd": "SGD",
        "sar": "SAR",
        "twd": "TWD",
        "lyd": "LYD",
        "iqd": "IQD",
        "sek": "SEK",
        "inr": "INR",
        "aud": "AUD",
        "egp": "EGP"
    }

    if x in ["unknown", "nan", "none", "n/a", "na", ""]:
        return pd.NA

    return currency_map.get(x, x.upper())

clean_df["currency"] = clean_df["currency"].apply(clean_currency)

print(clean_df["currency"].value_counts(dropna=False))

currency
ILS     1217
USD      409
GBP       54
<NA>      35
EUR       23
CHF       19
AED       10
JOD       10
SGD        7
EGP        6
SAR        4
TWD        2
LYD        1
IQD        1
SEK        1
AUD        1
INR        1
Name: count, dtype: int64


In [12]:
def clean_category(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()
    key = x.lower().replace(":", "").strip()

    category_map = {
        "electronics": "Electronics",
        "electronics and tech": "Electronics",
        "electronics and accessories": "Electronics",

        "furniture": "Furniture",

        "clothing and accessories": "Clothing and Accessories",
        "clothing & accessories": "Clothing and Accessories",
        "clothing": "Clothing and Accessories",
        "clothes": "Clothing and Accessories",

        "sports equipment": "Sports Equipment",
        "sports equipments": "Sports Equipment",
        "sports": "Sports Equipment",

        "toys and games": "Toys and Games",
        "toys & games": "Toys and Games",
        "games and toys": "Toys and Games",
        "toys": "Toys and Games",
        "games": "Toys and Games",

        "musical instruments": "Musical Instruments",
        "musical instrument": "Musical Instruments",

        "kitchen and household items": "Kitchen and Household Items",
        "kitchen and household item": "Kitchen and Household Items",
        "kitchen and household": "Kitchen and Household Items",
        "kitchen & household": "Kitchen and Household Items",
        "kitchen & household items": "Kitchen and Household Items",
        "kitchen items": "Kitchen and Household Items",
        "kitchen": "Kitchen and Household Items",

        "other used products": "Other Used Products",
        "other": "Other Used Products",

        "home decor": "Home Decor",
        "home décor": "Home Decor",
        "decor": "Home Decor",
        "art and decor": "Home Decor",

        "personal care": "Personal Care",
        "health & personal care": "Personal Care",
        "beauty and personal care": "Personal Care"
    }

    return category_map.get(key, x)

clean_df["category"] = clean_df["category"].apply(clean_category)

print("Categories before:", df["category"].nunique(dropna=True))
print("Categories after:", clean_df["category"].nunique(dropna=True))

Categories before: 132
Categories after: 75


In [15]:
text_columns = ["title", "description", "category", "currency", "item_id", "item_photo"]

for col in text_columns:
    if col in clean_df.columns:
        clean_df[col] = (
            clean_df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )


In [16]:
print("Final categories:")
print(clean_df["category"].value_counts(dropna=False))

print("\nFinal currencies:")
print(clean_df["currency"].value_counts(dropna=False))

Final categories:
category
Electronics                     384
Clothing and Accessories        253
Kitchen and Household Items     246
Furniture                       188
Books                           143
                               ... 
Refrigerators                     1
Dining Tables and Chair Sets      1
Electric Bicycles                 1
Acoustic Guitars                  1
Phone Accessories                 1
Name: count, Length: 76, dtype: Int64

Final currencies:
currency
ILS     1217
USD      409
GBP       54
<NA>      35
EUR       23
CHF       19
AED       10
JOD       10
SGD        7
EGP        6
SAR        4
TWD        2
LYD        1
IQD        1
SEK        1
AUD        1
INR        1
Name: count, dtype: Int64


Inconsistencies in category names, currency formats, and text representations were standardized.

Category labels representing the same category but differing in capitalization, spacing, punctuation, singular/plural forms, or equivalent wording were mapped to a consistent representation. This reduced the number of raw category representations from 132 to 75 distinct non-missing category labels. Semantically different categories were preserved rather than being arbitrarily merged into broader groups.

Currency values were standardized to consistent three-letter currency codes. For example, `NIS`, `Shekel`, `shekel`, and the shekel symbol were standardized to `ILS`, while `usd` and `dollar` were standardized to `USD`. Unknown or missing currency representations were converted to missing values.

Text fields were also standardized by removing leading and trailing whitespace and collapsing repeated spaces.